# ARK-5 Human Detector — YOLOv8 Training (Google Colab)

**Classes:** `standing` (0), `fallen` (1)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Prepare `human_dataset.zip` on laptop:
   ```bash
   python scripts/make_colab_zip.py
   ```
3. Upload zip in Cell 2

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable T4 GPU!')

In [ ]:
import zipfile, os
from google.colab import files

print('Upload human_dataset.zip from laptop...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print('Uploaded:', zip_name)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content')

DATA_DIR = '/content/human_dataset'
assert os.path.isfile(f'{DATA_DIR}/data.yaml'), 'data.yaml missing in zip!'

# FIX: data.yaml ke 'path'/'train'/'val' relative the (e.g. 'images/train'),
# jo Colab mein current working directory ke against resolve ho rahe the,
# DATA_DIR ke against nahi -- is se training ke waqt 'images not found'
# error aata tha. Ab hum train/val ko poora ABSOLUTE path bana dete hain
# taake koi ambiguity na rahe.
import yaml
_yaml_path = f'{DATA_DIR}/data.yaml'
with open(_yaml_path) as f:
    _d = yaml.safe_load(f)
_d['path'] = DATA_DIR
_d['train'] = f'{DATA_DIR}/images/train'
_d['val'] = f'{DATA_DIR}/images/val'
with open(_yaml_path, 'w') as f:
    yaml.safe_dump(_d, f)
print(open(_yaml_path).read())

train_imgs = os.listdir(f'{DATA_DIR}/images/train')
val_imgs = os.listdir(f'{DATA_DIR}/images/val')
print(f'Train images: {len(train_imgs)}')
print(f'Val images:   {len(val_imgs)}')


In [ ]:
# yolov8n = fast (recommended for ESP32-CAM stream on laptop)
# yolov8s = better accuracy, slower inference
model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{DATA_DIR}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    augment=True,
    project='/content/runs',
    name='human_detector',
    device=0,
)

# FIX: agar isi Colab session mein pehle bhi training chal chuki ho
# (jaise error ke baad dobara try), Ultralytics khud folder ka naam
# badal deta hai (human_detector -> human_detector2, ...). Hardcoded
# naam ke bajaye, results.save_dir se ASAL folder path uthate hain —
# taake Cell 4/5 hamesha sahi jagah dekhein, chahe kitni baar retrain
# kiya ho.
RUN_DIR = str(results.save_dir)
print('Run saved at:', RUN_DIR)
print('Training complete!')


In [ ]:
from IPython.display import Image, display
display(Image(f'{RUN_DIR}/results.png'))


In [ ]:
from google.colab import files
files.download(f'{RUN_DIR}/weights/best.pt')
print('Downloaded best.pt — copy to human_detector_robot/models/best.pt')


## After download

1. `best.pt` → `human_detector_robot/models/best.pt`
2. Update `config/project.yaml`:
   ```yaml
   yolo:
     weights: best.pt
     custom_model: true
   ```
3. Run: `python main_controller.py --no-motors`